# 情報数理 基礎Ⅰ演習 第6回｜果てのない面積を、極限で測る

スライドの札にあるセルを、上から順に実行します（セルを選んで `Shift` + `Enter`）。

- 🔹 のセルは、実行するだけです
- ✏️ のセルは、`____` を埋めてから実行します

コードは半角で打ちます。

## 0. 提出のしかた

1. Colab の左上で、ファイル名を suuri06_学籍番号.ipynb に変えます（学籍番号は半角で打ちます）。名前を変えられないときは、［ファイル］→［ドライブにコピーを保存］を選んでから変えます
2. ✏️ の 5 つのセルを埋めて、実行します
3. ［ファイル］→［ダウンロード］→［.ipynb をダウンロード］で保存します
4. Classroom の第6回の課題に添付して、［提出］を押します

## Python で広義積分を書く（🔹）

### 🔹 ノート P1 for で端を伸ばして表にする（スライド 5-1）

端 b を 10 倍ずつ伸ばし、1/x² の 1 から b までの面積 `1 - 1/b` と、1/x の 1 から b までの面積 `math.log(b)` を並べます。`math.log` に引数を 1 つ渡すと、自然対数（底が e）を返します。出力は、b・1/x² の面積・1/x の面積の順です。第5回の for の続きです。

In [ ]:
import math
for b in [10, 100, 1000, 10000]:
    print(b, 1 - 1/b, math.log(b))

**数を替えてみる**

`[10, 100, 1000, 10000, 100000]` に替えると、最後の行は `100000 0.99999 11.512925464970229` です。

### 🔹 ノート P2 while で「変わらなくなるまで」伸ばす（スライド 5-2）

`while True:` の中で b を 2 倍ずつ伸ばし、1/x² の面積の増え方 `diff` が目安 `tol` より小さくなったら、`break` で while を抜けます。50 回まわっても止まらないときも抜けます。出力は、b を 2 倍にした回数・b・面積・最後の増え方です。

In [ ]:
import math
tol = 0.001
b = 1
area = 1 - 1/b
count = 0
while True:
    b = b * 2
    new = 1 - 1/b
    diff = new - area
    area = new
    count = count + 1
    if diff < tol:
        break
    if count == 50:
        print("上限の 50 回で止めます")
        break
print(count, b, area, diff)

**数を替えてみる**

`tol = 0.0001` に替えると、出力は `14 16384 0.99993896484375 6.103515625e-05` です。目安を小さくすると、まわる回数が増えます。

### 🔹 ノート P3 SymPy の oo で広義積分を出す（スライド 5-3）

`oo` は SymPy の正の無限大で、`(x, 1, oo)` と書くと 1 から ∞ までの範囲になります。`limit(1 - 1/b, b, oo)` は、b を ∞ へ伸ばしたときの `1 - 1/b` の極限です。第5回の SymPy の続きです。

In [ ]:
from sympy import symbols, integrate, limit, sqrt, oo
x = symbols('x')
b = symbols('b', positive=True)
print(integrate(1/x**2, (x, 1, oo)))
print(integrate(1/x, (x, 1, oo)))
print(integrate(1/sqrt(x), (x, 0, 1)))
print(limit(1 - 1/b, b, oo))

**数を替えてみる**

最初の print の `1/x**2` を `1/x**5` に替えると、出力の 1 行目は `1/4` です。

### 🔹 ノート P4 SciPy の quad に np.inf を渡す（スライド 5-4）

`quad` の端に `np.inf`（NumPy の無限大）を渡すと、∞ までの範囲を数で計算します。返る組の 1 つ目は近似値、2 つ目は絶対誤差の推定値です。2 つ目の print の組の 1 つ目の数は、√π を float の計算で出した値（3 つ目の print の `math.sqrt(math.pi)`）と同じ字です。第5回の quad の続きです。

In [ ]:
import math
import numpy as np
from scipy.integrate import quad
print(quad(lambda x: 1/x**2, 1, np.inf))
print(quad(lambda x: np.exp(-x**2), -np.inf, np.inf))
print(math.sqrt(math.pi))

**数を替えてみる**

2 つ目の print の範囲を `0, np.inf` に替えると、出力の 2 行目は `(0.8862269254527579, 7.101318529726462e-09)` です。

### 🔹 ノート P5 特異点の手前で止まる計算機（スライド 5-5）

第5回の `riemann` を、1/√x の 0 から 1 までに当てます。短冊の高さは区間の真ん中の点で取ります。出力は、本数 n・短冊の和・2 との差です。最後の行は `quad` の組です。

In [ ]:
import numpy as np
from scipy.integrate import quad
def riemann(f, a, b, n):
    h = (b - a) / n
    total = 0
    for k in range(n):
        x = a + (k + 0.5) * h
        total += f(x) * h
    return total
for n in [10, 100, 1000, 10000]:
    s = riemann(lambda x: 1 / x**0.5, 0, 1, n)
    print(n, s, 2 - s)
print(quad(lambda x: 1/np.sqrt(x), 0, 1))

**数を替えてみる**

高さを取る点を `x = a + k * h`（左端）に替えると、`ZeroDivisionError` で止まります。最初の短冊の高さを x ＝ 0 で計算するためです。

## スライドの続き（🔹）

### 🔹 ノート P6 1/xᵖ の面積は、p と 1 の大小で決まる（スライド 6-2）

p を 1/2・1・2・3 にして、1/xᵖ の 1 から ∞ までの面積と、0 から 1 までの面積を並べます。1/2 は `Rational(1, 2)` で書きます（`0.5` と書くと、SymPy が止まります）。出力は、p・∞ の側・0 の側の順です。

In [ ]:
from sympy import symbols, integrate, oo, Rational
x = symbols('x')
for p in [Rational(1, 2), 1, 2, 3]:
    print(p, integrate(1/x**p, (x, 1, oo)), integrate(1/x**p, (x, 0, 1)))

**数を替えてみる**

リストの最後を 5 にして `[Rational(1, 2), 1, 2, 5]` に替えると、最後の行は `5 1/4 oo` です。

### 🔹 ノート P7 比べて見分ける・確率の全体は 1（スライド 6-4）

最初の print は、e^(−x²) と e^(−x) の 1 から ∞ までの面積を小数で出します。3 つ目と 4 つ目の print は、k を正の文字にして、k e^(−kx) の 0 から ∞ までの面積（全体）と、x を掛けた面積（平均）を出します。

In [ ]:
from sympy import symbols, integrate, exp, oo, N
x = symbols('x')
print(N(integrate(exp(-x**2), (x, 1, oo))), N(integrate(exp(-x), (x, 1, oo))))
print(integrate(exp(-x**2), (x, -oo, oo)))
k = symbols('k', positive=True)
print(integrate(k*exp(-k*x), (x, 0, oo)))
print(integrate(x*k*exp(-k*x), (x, 0, oo)))

**数を替えてみる**

2 つ目の print の `exp(-x**2)` を `exp(-x**2/2)` に替えると、出力の 2 行目は `sqrt(2)*sqrt(pi)` です。

## 演習（✏️）

`____` を埋めて実行します。ヒントは、スライド 7-1・7-2 にあります。

### ✏️ ノート 7-1 問 1

$\int_{2}^{\infty}\frac{3}{x^{4}}\,dx$ を求めます。

In [ ]:
from sympy import symbols, integrate, oo
x = symbols('x')
f = 3 / x**4
result = ____
print("広義積分:", result, float(result))

### ✏️ ノート 7-1 問 2

$\int_{0}^{9}\frac{dx}{\sqrt{x}\mathstrut}$ を求めます。

In [ ]:
from sympy import symbols, integrate, limit, sqrt
x = symbols('x')
eps = symbols('eps', positive=True)
F = integrate(1 / sqrt(x), (x, eps, 9))
result = ____
print("eps で止めた面積:", F)
print("広義積分:", result)

### ✏️ ノート 7-1 問 3

$\int_{1}^{\infty}x^{-4/3}\,dx$ が収束するかを調べ、収束するなら値を求めます。

In [ ]:
from sympy import symbols, integrate, oo, Rational
x = symbols('x')
p = ____
result = integrate(1 / x**p, (x, 1, oo))
print("p:", p)
print("広義積分:", result)

### ✏️ ノート 7-2 問 1

月の GM を $4.90 \times 10^{12}$ m³/s²、月の半径を 1,737,400 m とします。月の表面から重力の外へ出るのに 1 kg あたり要る仕事と、そのための速さ（km/s）を求めます。

In [ ]:
from sympy import symbols, integrate, oo, sqrt
r = symbols('r', positive=True)
GM = 4900000000000
R = 1737400
W = ____
v = sqrt(2 * W)
print("1 kg あたりの仕事（J）:", round(float(W)))
print("速さ（km/s）:", round(float(v) / 1000, 3))

### ✏️ ノート 7-2 問 2

年 90 万円を連続に受け取り続け、割引の速さ 0.025 で今の値に直す模型を考えます。終わりのない受け取り全体の今の値（万円）を求めます。

In [ ]:
from sympy import symbols, integrate, exp, oo, Rational
t = symbols('t', positive=True)
a = 90
k = Rational("0.025")
value = ____
print("今の値（万円）:", value)